In [0]:
# Remove duplicates

data = [("John", 28, "M"), ("Alice", 34, "F"), ("John", 28, "M")]
columns = ["name", "age", "gender"]

df = spark.createDataFrame(data, columns)
df = df.dropDuplicates(["age", "name"])
display(df)

In [0]:
data1 = [("1", "John"), ("2", "Alice"), ("3", "Bob")]
columns1 = ["ID", "Name"]

df1 = spark.createDataFrame(data1, columns1)
data2 = [("1", "HR"), ("2", "Finance"), ("4", "IT")]
columns2 = ["ID", "Department"]

df2 = spark.createDataFrame(data2, columns2)

# Inner join
inner_join = df1.join(df2, df1.ID == df2.ID, "inner")

# Left join
left_join = df1.join(df2, df1.ID==df2.ID, "left")

#show results
inner_join.display()
left_join.display()

In [0]:
# Calculate Average using Window functions

from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Sample DataFrame
data = [("Alice", 100), ("Bob", 200), ("Cathy", 150), ("David", 300)]
columns = ["Name", "Score"]

df = spark.createDataFrame(data, columns)

window_spec = Window.partitionBy("Score").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_with_avg= df.withColumn("running_avg", avg("Score").over(window_spec ))

df_with_avg.display()

In [0]:
from pyspark.sql.functions import to_date
from pyspark.sql import Window
from pyspark.sql import functions as F

In [0]:
# Daily user retention for a 30-day cohort window

data = [(1, '2023-01-01'), (1, '2023-01-02'), (2, '2023-01-01'), (2, '2023-01-03'), (3, '2023-01-02')]
df = spark.createDataFrame(data, ["user_id", "login_date"])
df = df.withColumn("login_date", to_date("login_date"))

cohort = df.groupBy("user_id").agg(F.min("login_date").alias("cohort_date"))

df = df.join(cohort, "user_id")
df = df.withColumn("days_since_cohort", F.datediff("login_date", "cohort_date"))

df  = df.groupBy("cohort_date", "days_since_cohort").agg(F.countDistinct("user_id").alias("retained_users"))

df.display()

In [0]:
# Latest 3 events per user
events = [(1, 'login', '2023-01-01 10:00'), (1, 'click', '2023-01-01 10:05'), (1, 'logout', '2023-01-01 10:10'),
          (1, 'login', '2023-01-02 10:00'), (2, 'login', '2023-01-01 09:00')]

df = spark.createDataFrame(events, ["user_id", "event", "ts"])
df = df.withColumn("ts", F.to_timestamp("ts"))

w= Window.partitionBy("user_id").orderBy(F.desc("ts"))

df = df.withColumn("rn", F.row_number().over(w)).filter("rn <=3")
df.display()